# Caso J · 04 Integración tráfico × meteorología — predicción congestión

> _Tutorial · Caso de uso: **J — Tráfico + YOLO** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Cruzar conteos de tráfico con meteorología (lluvia) y entrenar un modelo que prediga `congestion_level`.


## 2. Qué se aprende

- Merge time-aligned.
- Random Forest para clasificación ordinal.
- Feature importance climática.


## 3. Contexto del caso de uso

Capa oro: predicción para los próximos 15 min.


## 4. Relación con CENTINELA+

Tool del chatbot Caso H opcional.


## 5. Relación con Medallion

Lee plata, escribe oro.


## 6. Datos de entrada

Mock traffic + AEMET (incluido en mismo CSV).


## 7. Schema CAPTIA esperado

No aplica (oro).


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos.


In [ ]:
csv_path = ROOT / "notebooks/_data/traffic_camera_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.dayofweek
df["is_rush"] = ((df["hour"].isin([7, 8, 9, 17, 18, 19])) & (df["weekday"] < 5)).astype(int)
df.head()


## 10. Exploración paso a paso

Correlación.


In [ ]:
print(df[["vehicle_count", "precip_mm", "is_rush", "congestion_level"]].corr().round(2))


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Modelo.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = df[["vehicle_count", "precip_mm", "is_rush", "hour"]]
y = df["congestion_level"]
n = len(df); i = int(n * 0.7)
X_tr, X_te = X.iloc[:i], X.iloc[i:]
y_tr, y_te = y.iloc[:i], y.iloc[i:]

m = RandomForestClassifier(n_estimators=120, random_state=SEED).fit(X_tr, y_tr)
y_pred = m.predict(X_te)
print(classification_report(y_te, y_pred, zero_division=0))


## 13. Visualizaciones explicativas

Feature importance.


In [ ]:
imp = pd.Series(m.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(color="#9C27B0", figsize=(7, 3))
plt.title("Feature importance — congestion_level")
plt.tight_layout()


## 14. Validaciones

El modelo discrimina niveles base.


In [ ]:
from sklearn.metrics import balanced_accuracy_score
print({"balanced_acc": balanced_accuracy_score(y_te, y_pred)})


## 15. Errores comunes

1. Predecir como regresión cuando es ordinal.
2. No incluir hora del día.
3. Comparar accuracy entre datasets desbalanceados.


## 16. Ejercicios propuestos

1. Añade `dvehicle_count_15min` como feature.
2. Reemplaza por `OrdinalRegression`.
3. Construye un dashboard Grafana con prediction live.


## 17. Cómo se reutiliza con datos reales

Cambiar el mock por la query Flux que combina `traffic_cameras` + `weather_station`.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Documento web del caso: `docs/use-cases/case-j-traffic-yolo.md`.
- ¡Has completado los 42 notebooks didácticos!
- Vuelve a `00_project_overview/00_arquitectura_medallion_captia.ipynb` para revisitar el mapa.
